In [ ]:
import random
import torchaudio
import torch
from transformers import AutoConfig, AutoModel # for wav2vec
from transformers import WhisperProcessor # for whisper
from transformers import VoxtralRealtimeForConditionalGeneration, AutoProcessor # for voxtral
from mistral_common.tokens.tokenizers.audio import Audio
from tqdm import tqdm
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device available is', device)

seed = 7 
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# TOGGLE THIS ON IF YOU WANT TO RE-GENERATE THE EMBEDDING SETS
generate = True

## Get our encoders ready

In [ ]:
# wav2vec embedding wrapper

wav2vec_config = AutoConfig.from_pretrained("facebook/wav2vec2-base")
wav2vec = AutoModel.from_pretrained("facebook/wav2vec2-base", device_map="cuda")

def generate_wav2vec_embedding(waveform):
  # extract features
  emission = wav2vec(waveform.to(device)).last_hidden_state
  emission = emission.detach().cpu()
  # emission = emission.detach().cpu().numpy()
  return emission

In [ ]:
# whisper-small embedding wrapper

whisper_small_config = AutoConfig.from_pretrained("openai/whisper-small")
whisper_small_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_small = AutoModel.from_pretrained("openai/whisper-small", device_map="cuda")

def generate_whisper_small_embedding(input_features):

  # no clue why this is required but it was in the docs and throws and error if not included
  decoder_input_ids = torch.tensor([[1, 1]]) * whisper_small.config.decoder_start_token_id

  emission = whisper_small.encoder(
    input_features.to(device),
    decoder_input_ids=decoder_input_ids.to(device)
    ).last_hidden_state
  
  emission = emission.detach().cpu()
  return emission

In [ ]:
# whisper embedding wrapper

whisper_large_config = AutoConfig.from_pretrained("openai/whisper-large-v3")
whisper_large_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper_large = AutoModel.from_pretrained("openai/whisper-large-v3", device_map="cuda")

def generate_whisper_large_embedding(input_features):

  # no clue why this is required but it was in the docs and throws and error if not included
  decoder_input_ids = torch.tensor([[1, 1]]) * whisper_large.config.decoder_start_token_id

  emission = whisper_large.encoder(
    input_features.to(device),
    decoder_input_ids=decoder_input_ids.to(device)
    ).last_hidden_state
  
  emission = emission.detach().cpu()
  return emission

In [ ]:
# Voxtral embedding wrapper

voxtral_processor = AutoProcessor.from_pretrained("mistralai/Voxtral-Mini-4B-Realtime-2602")
voxtral = VoxtralRealtimeForConditionalGeneration.from_pretrained("mistralai/Voxtral-Mini-4B-Realtime-2602", device_map="cuda")

def generate_voxtral_embedding(inputs):
    emission = voxtral.model.audio_tower(
        input_features=inputs["input_features"].to(device),
        # return_dict=True
    ).last_hidden_state

    emission = emission.detach().cpu()
    return emission

## Set up timit data
First, we set up our timit data so that we have phone-audio pairings to train over.
`.phn` files in the corpus have triples of (sample beginning time, sample ending time, phone ARPABET label).

In [ ]:
# get data about train/test split
test_df = pd.read_csv('data/darpa-timit-acousticphonetic-continuous-speech/test_data.csv')
test_df = test_df[test_df['is_audio'] == True]
test_df = test_df[test_df['filename'].str.contains('.wav')] # for some reason, the wav files have both .WAV and .WAV.wav files. Only the .WAV.wav files are readable, so we filter the others out

train_df = pd.read_csv('data/darpa-timit-acousticphonetic-continuous-speech/train_data.csv')
train_df = train_df[train_df['is_audio'] == True]
train_df = train_df[train_df['filename'].str.contains('.wav')] # for some reason, the wav files have both .WAV and .WAV.wav files. Only the .WAV.wav files are readable, so we filter the others out

# helper function
def get_phn_file(audio_path):
    part = audio_path.split('.')[0] # the part before the extension
    return part + '.PHN'


In [ ]:
# embedding helpers

def get_wav2vec_embedding(audio_path):
    # returns a list of embeddings and a list with the corresponding phone labels
    phone_path = get_phn_file(audio_path)
    loaded = []
    labels = []
    
    with open(phone_path) as file:
        for line in file:
            begin, end, phone = line.split()
            begin = int(begin)
            end = int(end)

            if phone == '#h' or phone == 'h#': # file boundary
                pass

            else:
                num_frames = end - begin
                if num_frames > 400: # otherwise wav2vec freaks out

                    audio, sample_rate = torchaudio.load(audio_path, frame_offset=begin, num_frames=num_frames)

                    with torch.no_grad():
                        embedding = generate_wav2vec_embedding(audio)
                    embedding = torch.mean(embedding, dim=1) # average over all frames the phone is present for
                    
                    shape = embedding.shape
                    if shape[1] == 768:
                        embedding = embedding.squeeze()
                        loaded.append(embedding)
                        labels.append(phone)
    
    return loaded, labels

def get_whisper_embedding(audio_path):
    # returns a list of embeddings and a list with the corresponding phone labels
    phone_path = get_phn_file(audio_path)
    loaded = []
    labels = []
    
    with open(phone_path) as file:
        for line in file:
            begin, end, phone = line.split()
            begin = int(begin)
            end = int(end)

            if phone in ['h#', 'epi', 'pau', '1', '2']: # non-phone info
                pass

            else:
                num_frames = end - begin
                if num_frames > 400: # otherwise wav2vec freaks out

                    audio, sample_rate = torchaudio.load(audio_path, frame_offset=begin, num_frames=num_frames)

                    audio = whisper_small_processor(
                        audio.squeeze().numpy(),
                        sampling_rate=16000,
                        return_tensors="pt"
                    )

                    # input_features = audio["input_features"].to(dtype=torch.float16).to(device) # otherwise whisper freaks out
                    input_features = audio["input_features"].to(device)

                    with torch.no_grad():
                        embedding = generate_whisper_small_embedding(input_features)
                    embedding = torch.mean(embedding, dim=1) # average over all frames the phone is present for
                    
                    shape = embedding.shape
                    # print(shape)
                    if shape[1] == 768:
                        embedding = embedding.squeeze()
                        loaded.append(embedding)
                        labels.append(phone)
    
    return loaded, labels

# voxtral helper
def get_voxtral_embedding(audio_path):
    # returns a list of embeddings and a list with the corresponding phone labels
    phone_path = get_phn_file(audio_path)
    loaded = []
    labels = []
    
    with open(phone_path) as file:
        for line in file:
            begin, end, phone = line.split()
            begin = int(begin)
            end = int(end)

            if phone in ['h#', 'epi', 'pau', '1', '2']: # non-phone info
                pass

            else:
                num_frames = end - begin
                if num_frames > 400: # otherwise wav2vec freaks out

                    audio = Audio.from_file(audio_path) #TODO: instead of from_file, use torch audio load and have specifically the phone
                    audio.resample(voxtral_processor.feature_extractor.sampling_rate)

                    inputs = voxtral_processor(audio.audio_array, return_tensors="pt")
                    inputs = inputs.to(voxtral.device, dtype=voxtral.dtype)

                    with torch.no_grad():
                        embedding = generate_voxtral_embedding(inputs)
                    embedding = torch.mean(embedding, dim=1) # average over all frames the phone is present for
                    
                    shape = embedding.shape
                    # print(shape)
                    if shape[1] == 1280:
                        embedding = embedding.squeeze()
                        loaded.append(embedding)
                        labels.append(phone)
    
    return loaded, labels

In [ ]:
# generate our whisper embeddings

if generate:
    whisper_train_embeddings = []
    whisper_train_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(train_df.itertuples(), "Generating train whisper embeddings", total=train_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_whisper_embedding(audio_path)

        whisper_train_embeddings.extend(x)
        whisper_train_labels.extend(y)

    # save to disk
    torch.save(whisper_train_embeddings, 'data/saved_embeddings/whisper_train_embeddings.pt')
    torch.save(whisper_train_labels, 'data/saved_embeddings/whisper_train_labels.pt')

    # empty previous variables
    whisper_train_embeddings = None
    whisper_train_labels = None

In [ ]:
if generate:
    whisper_test_embeddings = []
    whisper_test_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(test_df.itertuples(), "Generating test whisper embeddings", total=test_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_whisper_embedding(audio_path)

        whisper_test_embeddings.extend(x)
        whisper_test_labels.extend(y)

    # save to disk
    torch.save(whisper_test_embeddings, 'data/saved_embeddings/whisper_test_embeddings.pt')
    torch.save(whisper_test_labels, 'data/saved_embeddings/whisper_test_labels.pt')

    # empty previous variables
    whisper_test_embeddings = None
    whisper_test_labels = None

In [ ]:
# generate our wav2vec train embeddings

if generate:
    wav2vec_train_embeddings = []
    wav2vec_train_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(train_df.itertuples(), "Generating train wav2vec embeddings", total=train_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_wav2vec_embedding(audio_path)

        wav2vec_train_embeddings.extend(x)
        wav2vec_train_labels.extend(y)

    # save to disk
    torch.save(wav2vec_train_embeddings, 'data/saved_embeddings/wav2vec_train_embeddings.pt')
    torch.save(wav2vec_train_labels, 'data/saved_embeddings/wav2vec_train_labels.pt')

    # empty previous variables
    wav2vec_train_embeddings = None
    wav2vec_train_labels = None

In [ ]:
# generate our wav2vec test embeddings

if generate:
    wav2vec_test_embeddings = []
    wav2vec_test_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(test_df.itertuples(), "Generating test wav2vec embeddings", total=test_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_wav2vec_embedding(audio_path)

        wav2vec_test_embeddings.extend(x)
        wav2vec_test_labels.extend(y)

    # save to disk
    torch.save(wav2vec_test_embeddings, 'data/saved_embeddings/wav2vec_test_embeddings.pt')
    torch.save(wav2vec_test_labels, 'data/saved_embeddings/wav2vec_test_labels.pt')

    # empty previous variables
    wav2vec_test_embeddings = None
    wav2vec_test_labels = None

In [ ]:
# generate our voxtral train embeddings

if generate:
    voxtral_train_embeddings = []
    voxtral_train_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(train_df.itertuples(), "Generating train voxtral embeddings", total=train_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_voxtral_embedding(audio_path)

        voxtral_train_embeddings.extend(x)
        voxtral_train_labels.extend(y)

    # save to disk
    torch.save(voxtral_train_embeddings, 'data/saved_embeddings/voxtral_train_embeddings.pt')
    torch.save(voxtral_train_labels, 'data/saved_embeddings/voxtral_train_labels.pt')

    # empty previous variables
    voxtral_train_embeddings = None
    voxtral_train_labels = None

In [ ]:
# generate our voxtral test embeddings

if generate:
    voxtral_test_embeddings = []
    voxtral_test_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(test_df.itertuples(), "Generating test voxtral embeddings", total=test_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_voxtral_embedding(audio_path)

        voxtral_test_embeddings.extend(x)
        voxtral_test_labels.extend(y)

    # save to disk
    torch.save(voxtral_test_embeddings, 'data/saved_embeddings/voxtral_test_embeddings.pt')
    torch.save(voxtral_test_labels, 'data/saved_embeddings/voxtral_test_labels.pt')

    # empty previous variables
    voxtral_test_embeddings = None
    voxtral_test_labels = None

Generating test voxtral embeddings:  14%|█▍        | 242/1680 [04:32<26:06,  1.09s/it]